# X03 RDD2022 pilot: prepare
Private research run. Japan is not used.

In [ ]:
from pathlib import Path
Path('prepare.py').write_text('"""CPU-only original RDD2022 preparation; no Japan access."""\nimport collections\nimport hashlib\nimport io\nimport json\nimport random\nimport shutil\nimport tarfile\nimport time\nimport urllib.request\nimport zipfile\nimport xml.etree.ElementTree as ET\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\nCOUNTRIES = {\'India\': 7706, \'Czech\': 2829, \'United_States\': 4805}\nCLASSES = [\'D00\', \'D10\', \'D20\', \'D40\']\nBASE = \'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/\'\n\nclass RemoteArchive(io.RawIOBase):\n    """Read only requested ZIP ranges from the official Figshare archive."""\n    def __init__(self):\n        import requests\n        self.session=requests.Session()\n        response=self.session.get(\'https://ndownloader.figshare.com/files/38030910?x03=\'+str(time.time_ns()),headers={\'Range\':\'bytes=0-127\'},timeout=60)\n        response.raise_for_status(); assert response.status_code==206\n        self.url=response.url; self.size=int(response.headers[\'Content-Range\'].split(\'/\')[-1]); self.pos=0\n    def seekable(self): return True\n    def readable(self): return True\n    def tell(self): return self.pos\n    def seek(self,offset,whence=0):\n        self.pos=offset if whence==0 else self.pos+offset if whence==1 else self.size+offset\n        return self.pos\n    def read(self,n=-1):\n        if n<0: n=self.size-self.pos\n        n=min(n,self.size-self.pos)\n        if not n: return b\'\'\n        # Figshare signatures expire after ten seconds: obtain a fresh redirect per request.\n        r=self.session.get(\'https://ndownloader.figshare.com/files/38030910?x03=\'+str(time.time_ns()),headers={\'Range\':f\'bytes={self.pos}-{self.pos+n-1}\'},timeout=120)\n        r.raise_for_status()\n        if r.status_code!=206 or len(r.content)!=n: raise ValueError(\'Range server did not return requested bytes\')\n        self.pos+=n; return r.content\n\ndef sha(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for b in iter(lambda: f.read(1024 * 1024), b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef convert_box(coords, width, height):\n    # VOC inclusive 1-based corners -> continuous 0-based image-edge coordinates.\n    a, b, c, d = coords\n    a, b, c, d = max(0, a-1), max(0, b-1), min(width, c), min(height, d)\n    if c <= a or d <= b: raise ValueError(\'degenerate box\')\n    return [(a+c)/2/width, (b+d)/2/height, (c-a)/width, (d-b)/height]\n\ndef dhash(im):\n    a = np.asarray(im.convert(\'L\').resize((9, 8)), dtype=np.int16)\n    return int(\'\'.join(map(str, (a[:, 1:] > a[:, :-1]).astype(int).ravel())), 2)\n\ndef prepare(root=Path(\'/kaggle/working\')):\n    root = Path(root); raw = root/\'raw\'; out = root/\'prepared\'; out.mkdir(exist_ok=True)\n    (out/\'images\').mkdir(exist_ok=True); (out/\'labels\').mkdir(exist_ok=True)\n    records, errors, provenance = [], [], []\n    # Original S3 country URLs now return 403. Official Figshare distributes a ZIP of country ZIPs.\n    remote=RemoteArchive(); outer=zipfile.ZipFile(remote)\n    print(\'Official archive members:\',outer.namelist(),flush=True)\n    for country, expected in COUNTRIES.items():\n        z = root/f\'RDD2022_{country}.zip\'\n        member=next(n for n in outer.namelist() if Path(n).name==country+\'.zip\')\n        url = \'https://figshare.com/articles/dataset/21431547 ; file 38030910 ; member \'+member\n        print(\'Range-download\', url, flush=True)\n        with outer.open(member) as src, open(z,\'wb\') as dst:\n            shutil.copyfileobj(src,dst,length=8*1024*1024)\n        provenance.append({\'country\': country, \'url\': url, \'bytes\': z.stat().st_size, \'sha256\': sha(z)})\n        with zipfile.ZipFile(z) as f:\n            # Extract only labelled training material; never challenge test data.\n            for member in f.infolist():\n                p = Path(member.filename)\n                if \'..\' in p.parts or p.is_absolute(): raise ValueError(\'unsafe archive\')\n                if \'train\' in [x.lower() for x in p.parts] and p.suffix.lower() in [\'.jpg\', \'.jpeg\', \'.xml\']:\n                    f.extract(member, raw/country)\n        xmls = {p.stem: p for p in (raw/country).rglob(\'*.xml\')}\n        images = sorted((raw/country).rglob(\'*.jpg\'))\n        if len(images) != expected: raise ValueError(f\'{country}: {len(images)} images, expected {expected}\')\n        for image in images:\n            try:\n                if image.stem not in xmls: raise ValueError(\'missing XML is not a negative\')\n                xml = xmls[image.stem]; tree = ET.parse(xml).getroot()\n                with Image.open(image) as im:\n                    im.load(); w, h = im.size; ph = dhash(im)\n                    pixels = hashlib.sha256(im.convert(\'RGB\').tobytes()).hexdigest()\n                if (int(tree.findtext(\'size/width\')), int(tree.findtext(\'size/height\'))) != (w, h):\n                    raise ValueError(\'XML/image dimensions disagree\')\n                boxes = []; ignored = []\n                for obj in tree.findall(\'object\'):\n                    name = obj.findtext(\'name\')\n                    if name not in CLASSES: ignored.append(name); continue\n                    coords = [float(obj.findtext(\'bndbox/\'+k)) for k in [\'xmin\',\'ymin\',\'xmax\',\'ymax\']]\n                    if not np.isfinite(coords).all(): raise ValueError(\'nonfinite coordinates\')\n                    boxes.append([CLASSES.index(name), *convert_box(coords, w, h)])\n                # Other real damage classes are outside this four-class task; retain images.\n                # A malformed spelling resembling a target class is quarantined for audit.\n                if any(n not in [\'D01\',\'D11\',\'D43\',\'D44\',\'D50\'] for n in ignored):\n                    raise ValueError(\'unrecognised class spelling: \'+str(ignored))\n                dest = out/\'images\'/image.name; shutil.copy2(image, dest)\n                (out/\'labels\'/f\'{image.stem}.txt\').write_text(\'\\n\'.join(\' \'.join(map(str,b)) for b in boxes))\n                records.append(dict(id=image.stem, country=country, width=w, height=h, boxes=boxes,\n                                    sha256=sha(image), xml_sha256=sha(xml), pixels=pixels, dhash=ph,\n                                    ignored_classes=ignored,\n                                    source_image=str(image.relative_to(raw)), source_xml=str(xml.relative_to(raw))))\n            except Exception as e: errors.append({\'image\': image.name, \'error\': str(e)})\n        z.unlink()\n    # Connected components: exact decoded duplicates and conservative dHash candidates <=3 bits.\n    # Candidate grouping is conservative leakage prevention, not proof of semantic duplication.\n    parent = list(range(len(records)))\n    def find(i):\n        while parent[i] != i: parent[i] = parent[parent[i]]; i = parent[i]\n        return i\n    def union(i,j): parent[find(i)] = find(j)\n    buckets = collections.defaultdict(list); pixels_seen = {}; pairs = []\n    for i,r in enumerate(records):\n        if r[\'pixels\'] in pixels_seen: union(i,pixels_seen[r[\'pixels\']])\n        pixels_seen[r[\'pixels\']] = i\n        candidates = set()\n        for chunk in range(4): candidates.update(buckets[(chunk,(r[\'dhash\']>>(16*chunk))&65535)])\n        for j in candidates:\n            if (r[\'dhash\'] ^ records[j][\'dhash\']).bit_count() <= 3:\n                union(i,j); pairs.append([records[j][\'id\'],r[\'id\']])\n        for chunk in range(4): buckets[(chunk,(r[\'dhash\']>>(16*chunk))&65535)].append(i)\n    groups = collections.defaultdict(list)\n    for i,r in enumerate(records): groups[find(i)].append(r)\n    for members in groups.values():\n        gid = min(r[\'id\'] for r in members)\n        for r in members: r[\'group\'] = gid\n        if any(r[\'country\']==\'United_States\' for r in members):\n            for r in members: r[\'split\'] = \'unseen\' if r[\'country\']==\'United_States\' else \'excluded_overlap\'\n    rng = random.Random(42)\n    source_groups = [m for m in groups.values() if \'split\' not in m[0]]\n    rng.shuffle(source_groups)\n    totals = collections.Counter(r[\'country\'] for m in source_groups for r in m)\n    assigned = collections.defaultdict(collections.Counter)\n    for members in sorted(source_groups, key=len, reverse=True):\n        counts = collections.Counter(r[\'country\'] for r in members)\n        split = max([\'train\',\'selection\',\'source_audit\'], key=lambda s:\n            sum(counts[c]*({\'train\':.8,\'selection\':.1,\'source_audit\':.1}[s]*totals[c]-assigned[c][s]) /\n                max(1,{\'train\':.8,\'selection\':.1,\'source_audit\':.1}[s]*totals[c]) for c in counts))\n        for r in members: r[\'split\'] = split; assigned[r[\'country\']][split] += 1\n    for split in [\'train\',\'selection\',\'source_audit\',\'unseen\']:\n        ids = [r[\'id\'] for r in records if r[\'split\']==split]\n        (out/f\'{split}.txt\').write_text(\'\\n\'.join(\'/kaggle/working/prepared/images/\'+i+\'.jpg\' for i in ids))\n    (out/\'manifest.json\').write_text(json.dumps(records))\n    audit = dict(counts={c:dict(collections.Counter(r[\'split\'] for r in records if r[\'country\']==c)) for c in COUNTRIES},\n                 quarantine=errors, near_duplicate_pairs=pairs, provenance=provenance,\n                 sequence_limitation=\'No verified route/sequence metadata; filename adjacency is not assumed to identify sequences.\',\n                 coordinate_convention=\'VOC inclusive 1-based corners; xmin/ymin minus one, bounds clipped to image edges\',\n                 duplicate_policy=\'Group all dHash<=3 candidates conservatively; exclude sources sharing a group with US. No manual duplicate certification.\')\n    (out/\'audit.json\').write_text(json.dumps(audit,indent=2))\n    for c in COUNTRIES:\n        subset = [r for r in records if r[\'country\']==c][:8]\n        canvas = Image.new(\'RGB\',(640,160*len(subset)),\'white\')\n        for k,r in enumerate(subset):\n            im = Image.open(out/\'images\'/f"{r[\'id\']}.jpg").convert(\'RGB\'); draw=ImageDraw.Draw(im)\n            for cls,x,y,w,h in r[\'boxes\']:\n                draw.rectangle([(x-w/2)*im.width,(y-h/2)*im.height,(x+w/2)*im.width,(y+h/2)*im.height],outline=\'red\',width=3)\n            im.thumbnail((640,150)); canvas.paste(im,(0,k*160))\n        canvas.save(out/f\'conversion_{c}.jpg\')\n    # All splits must remain group-disjoint, except excluded overlap records.\n    seen={}\n    for r in records:\n        if r[\'split\']==\'excluded_overlap\': continue\n        if r[\'group\'] in seen: assert seen[r[\'group\']]==r[\'split\']\n        seen[r[\'group\']]=r[\'split\']\n    with tarfile.open(root/\'prepared.tar.gz\',\'w:gz\') as f: f.add(out,arcname=\'prepared\')\n    print(json.dumps({k:v for k,v in audit.items() if k not in [\'near_duplicate_pairs\',\'quarantine\']},indent=2),flush=True)\n    # Preserve original XML for reproducibility; avoid saving duplicate raw images in output.\n    with tarfile.open(root/\'original_xml.tar.gz\',\'w:gz\') as f:\n        for p in raw.rglob(\'*.xml\'): f.add(p,arcname=str(p.relative_to(raw)))\n    shutil.rmtree(raw)\n    shutil.rmtree(out/\'images\'); shutil.rmtree(out/\'labels\')\n\nif __name__==\'__main__\': prepare()\n')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-u', 'prepare.py'], check=True)